In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from nanodrz import data
import torchaudio
import torch
from nanodrz.utils import play, resample, visualise_annotation
torch.cuda.set_device("cuda:1")

In [ ]:
audio, sr = torchaudio.load("/home/harry/test.wav")
audio = audio.sum(dim=0)[None]
audio = audio / audio.abs().max()
audio = resample(sr, 16000, audio)
audio = audio[:,:16000*25]

In [ ]:
from denoiser import pretrained
from denoiser.demucs import Demucs
import torch

denoiser =  torch.compile(pretrained.dns64().cpu().eval())

@torch.inference_mode()
def denoise(denoiser, audio, sr=None):
    audio = audio.sum(dim=0, keepdim=True)
    return denoiser(audio.cuda()).cpu()


In [ ]:
from nanodrz.model import DiarizeGPT, Config

device = torch.device("cuda:1")
torch.cuda.set_device(device)
ckpt = torch.load("/home/harry/runs/nanodrz/1708087744/0014000.pt", map_location=device)
config = Config(**ckpt["config"])
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
nlabels = model.generate(audio[:,:20*16000].cuda(), temperature=.8, max_steps=3*10)
visualise_annotation(nlabels)
play(audio[:,:20*16000])